ESERCIZIO GUIDATO: SENTIMENT ANALYSIS CON LSTM

Insegnare ad una macchina a sentire il tono di un discorso.
Il cervello filtra il rumore per concentrarsi su parole come felice, deluso, contento ed altro
Un modello che sa dove guardare grazie all'attenzione

- Tecniche di pulizia del testo 
- Architettura avanzata con layer LSTM e maccanismo di attenzione semplificato
- Integrazione del modello per inferenza su input utente in tempo reale.

Cominciamo dalle basi.
Perchè non possiamo dare il testo così com'è alla rete?

Pre-elaborazione del Linguaggio
Dalla parola grezza al segnale pulito
Il testo naturale è intrinsecamente rumoroso. Per una Sentiment Analysis efficace, non è sufficiente tokenizzare: dobbiamo isolare il nucleo informatio scartando gli elementi grammaticali superflui.
Se dico 'il film non è stato abbastanza male' il cuore della frase non è 'il' 'è' ma la relazione tra 'non' e 'male'
Dobbiamo standarizzare la frase affichè la rete non si perda in dettagli semantici innutili.
Trasformiamo queste fasi complesse in una sequenza standarizzata di operazioni che facilitano l'apprendimento delle relazioni semantiche da parte della rete neurale.

Ma quali sono gli strumenti che abbiamo per questa pulizia?

I Pilastri della Pulizia
Isolare il significato
* Stopwords Removal: eliminazione di termini comuni come articoli e preposizioni che non aggiungono valore al tono emotico.
* Lemmatizzazione: riduzione della parole alla loro radice morfologica o lemma per raggruppare varianti della stessa parola.
* Normalizzazione: conversione in minuscolo e rimozione di punteggiatura o caratteri speciali tramite espressioni regolari
* Tokenizzazione: suddivisione della stringa pulita in unità discrete chiamate token per la successiva vettorizzazione.

Immagina di dover catalogare dei libri, invece di leggere tutto crei degli indici di parole chiave standarizzate.

Approfondiamo i concetti di Stopwords e Lemmi

Stopwords e Lemmi
Parole come 'e' 'il' 'o' 'di' (parole neutre) compaiono con frequenza elevata ma sono neutre. La loro rimozione riduce la dimensionalità del vocabolario senza perdita di informazione (rumore linguistico)
A differenza dello stemming, la lemmatizzazione usa un'analisi morfologica completa per distinguere, ad esempio, tra 'vado' e 'andare', unificandoli correttamente (vantagigo della lemmatizzazione)
L'analisi della distribuzione dei termini permette di identificare parole rare o troppo comuni che potrebbero distorcere il peso statistico (frequenza di termini)

Vediamo come rendere questo processo fluido ed automatico

Pipeline di Preprocessing
Automazione del flusso
Non possiamo pulire ogni frase a mano, usiamo libreria apposite per creare una catena di montaggio.
Librerie come 'nltk' o 'spaCy' permettono di creare pipeline riutilizzabili che garantiscono consistenza tra i dati di addestramento e quelli di test.
Un preprocessing errato può portare il modello a imparare correlazioni spurie basate su punteggiatura o congiunzione anzichè sul reale sentimento.

Una volta puliti i dati possiamo passare all'architettura ed introduciamo l'Attenzione

LSTM con Attenzione
Ponderare le parole chiave
Mentre una LSTM standard analizza la sequenza in ordine, il meccaniscmo di attenzione permette alla rete di soffermarsi maggiormente su termini specifici come 'orribile' o 'eccellente'

Integriamo un layer di attenzione semplificato per pesare dinamicamente gli stati nascosti della LSTM

Ma come fa la matematica ad evidenziare un concetto?

La magia ridiede nei pesi
Meccanismo di Peso
Oltre l'ultimo stato nascosto
- Hidden Stats: ogni passo temporale della LSTM produce un vettore che rappresenta il contesto locale del token processato
- Attention Scores: calcolo di un valore di rilevanza per ogni istante temporale basato sulla compatibilità tra gli stati
- Context Vector: somma pesata di tutti gli stati nascosti che riassume l'intera sequenza focalizzandosi sulle parti importanti. riassunto intelligente dell'intera frase
- Interpretazione: il layer di attenzione permette di visualizzare quali parole hanno influenzato maggiormente la decisoine finale.

Dinamica del Layer
Allenamento:
    l'attenzione calcola un punteggio di allineamento tra uno stato query e gli stati chiave delle sequenza
Softmax di attenzione:
    i punteggi vengono normalizzati affinchè la loro somma sia unitaria, creando una distribuzione di probabilità sull'importanza del token
Output della Memoria
    Il vettore di contesto finale viene passato a un layer denso per la classificazione binaria o multiclasse del sentiment

Vantaggio della Ponderazione
Risolve il collo di bottiglia
Senza attenzione, la LSTM deve comprimere tutto il significato nell'ultimo stato (lstm classica). Questo causa perdita di informazione nelle frasi molto lunghe.
L'attenzione agisce come una memoria ad accesso diretto che permette di recuperare dettagli specifici ovunque si trovino nella sequenza.

Inferenza in Tempo Reale.
Dal modello all'applicazione
Un modello è utile solo se può essere interrogato su nuovi dati.
L'inferenza in tempo reale richiede di poter trasformare una stringa di caratteri digitati su una tastiera in una serie di segnali elettrici automatici. Processo che deve essere istantaneo.
Vedremo come gestire il flusso che trasforma una stringa inserita dall'utente in una predizione numerica.

Pipeline di Produzione
Dalla tastiera al risultato
- Input Processing: applicazione degli stessi filtri di pulizia usati durante il training per evitare discrepanze
- Vettorizzazione: conversione del testo in indici numerici utilizzando il vocabolario salvato precedentemente.
- Sequence Padding: garanzia che l'input utente abbia la stessa lunghezza richiesta dal modello tramite aggiunta di zeri.
- Probability Threshold: interpretazione dell'output sigmoideo per classificare il sentimento come positivo o negativo.

.keras è il formato standar per salvare non solo i pesi ma anche la struttura del layer di attenzione custom che abbiamo craeto, non basta salvare il modello ma anche il tokenizzer, il dizionario che trasforma le parole in numero.

Gestione del Modello
- Salvataggio e Caricamento: oltre ai pesi del modello, è fondamentale salvare i 'tokenizer' o gli oggetti di preprocessing per replicare l'indicizzazione
- Latenza di Risposta: in tempo reale, la velocità di inferenza è critica. Architetture leggere come la LSTM con attenzione sono ideali per risposte istantanee
- Classificazione Binaria: l'output finale rappresenta la confidenza del modello nell'appartennenza della classe positiva.

C'è un altro aspetto da considerare: l'ambiguità umana
Il linguaggio umano è pieno di parole ambigue, di sfumature. Cosa succese se il modello restituisce 0.5, il modello è confuso.


In [1]:
import nltk

print(nltk.__version__)

3.10.1


#python -m pip install --upgrade nltk   

In [2]:
import os
 
# 1. CONFIGURAZIONE BACKEND E IMPORTS
# In Keras 3 (2025/2026), l'impostazione del backend deve avvenire PRIMA di importare keras.
# Usiamo "torch" per sfruttare l'ecosistema PyTorch (ottimo per GPU AMD/NVIDIA) pur mantenendo l'API Keras.
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Risorse NLTK: necessarie per istruire i moduli linguistici su come gestire il dizionario inglese.
nltk.download('stopwords', quiet=True) # Scarica parole comuni "vuote" (the, is, at)
nltk.download('wordnet', quiet=True)   # Scarica il database lessicale per la lemmatizzazione
nltk.download('omw-1.4', quiet=True)   # Supporto multilingua per WordNet

# 2. PIPELINE DI PULIZIA DEL TESTO (Chapter 1)
def clean_text(text):
    """
    Funzione di Preprocessing: trasforma stringhe umane in input matematici stabili.
    """
    # re.sub(r'[^\w\s]', '', text.lower()): 
    # 1. Converte in minuscolo (perché "Ottimo" e "ottimo" devono avere lo stesso peso statistico).
    # 2. Rimuove la punteggiatura tramite espressione regolare (i simboli non portano sentiment).
    text = re.sub(r'[^\w\s]', '', text.lower())
    
    # Rimozione Stopwords: 
    # Filtriamo le parole ad alta frequenza che non influenzano il significato emotivo.
    stop_words = set(stopwords.words('english'))
    tokens = text.split() # Suddivide la stringa in una lista di singole parole
    tokens = [t for t in tokens if t not in stop_words] # Mantiene solo i termini "pesanti"
    
    # Lemmatizzazione:
    # Riduce la complessità del vocabolario trasformando ogni parola nella sua forma base (lemma).
    # Esempio: "Better" -> "Good". Questo riduce la dimensionalità dello spazio di embedding.
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return " ".join(tokens) # Ricombina i token in una stringa pulita



# 3. LAYER DI ATTENZIONE SEMPLIFICATA (Chapter 2)
@keras.saving.register_keras_serializable() # Permette a Keras di salvare il layer custom nel file .keras
class SimpleAttention(keras.layers.Layer):
    """
    Implementazione del meccanismo di Attention: permette alla rete di focalizzarsi
    sulle parole chiave (es. "fantastico") ignorando il contesto neutro.
    """
    def __init__(self, **kwargs):
        # Chiama il costruttore della classe base per gestire nomi e parametri
        super().__init__(**kwargs)

    def build(self, input_shape):
        # Inizializzazione dei pesi: creiamo un vettore W che imparerà l'importanza di ogni dimensione
        # shape=(input_shape[-1], 1): un peso per ogni feature dell'output dell'ultimo layer
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform", trainable=True)
        super().build(input_shape)

    def call(self, x):
        # 1. Calcolo Score: Eseguiamo il prodotto scalare tra l'input (x) e i pesi (W)
        # Formula: $e = x \cdot W$
        e = keras.ops.dot(x, self.W)
        
        # 2. Normalizzazione: Trasformiamo i punteggi in probabilità che sommano a 1
        # Formula: $\alpha = \frac{e^{e_i}}{\sum e^{e_j}}$ (Softmax)
        alpha = keras.ops.softmax(e, axis=1)
        
        # 3. Context Vector: Moltiplichiamo l'input originale per i pesi di attenzione
        # Il risultato è un vettore che "riassume" la sequenza dando più importanza ai token salienti.
        context = keras.ops.sum(x * alpha, axis=1)
        return context



# 4. DEFINIZIONE DEL MODELLO
vocab_size = 5000 # Numero massimo di parole uniche nel dizionario
max_len = 20      # Lunghezza fissa delle frasi (tronca o aggiunge zeri)

model = keras.Sequential([
    # Input Layer: Definisce la forma del batch di ingresso (20 numeri interi per frase)
    keras.layers.Input(shape=(max_len,)),
    
    # Embedding: Trasforma numeri discreti in vettori densi di 64 dimensioni.
    # Teoria: Le parole simili ("felice", "gioioso") verranno posizionate vicine nello spazio.
    keras.layers.Embedding(input_dim=vocab_size, output_dim=64),
    
    # LSTM: Memoria a lungo termine. 
    # return_sequences=True: Necessario perché il layer Attention deve "vedere" tutti gli stati della frase.
    keras.layers.LSTM(64, return_sequences=True),
    
    # Attention: Riduce la sequenza LSTM in un singolo vettore di contesto intelligente.
    SimpleAttention(),
    
    # Dropout: Spegne il 30% dei neuroni a ogni passo per evitare che la rete "impari a memoria".
    keras.layers.Dropout(0.3),
    
    # Dense: Output finale. Sigmoide schiaccia il risultato tra 0 e 1.
    keras.layers.Dense(1, activation='sigmoid')
])

# Ottimizzazione AdamW: Una variante di Adam che corregge il decadimento dei pesi (Weight Decay).
# Molto più efficace per modelli NLP che tendono a divergere.
model.compile(optimizer="adamw", loss="binary_crossentropy", metrics=["accuracy"])

# 5. SALVATAGGIO PROFESSIONALE
# Il formato .keras (V3) salva l'architettura (incluso il codice Python del layer Attention) e i pesi.
model.save("sentiment_model_v1.keras")
print("[INFO] Modello salvato correttamente in formato .keras")

# 6. TEST IN TEMPO REALE (Chapter 3)
def predict_sentiment(user_input, model, tokenizer_dict):
    """
    Funzione di inferenza: trasforma il testo dell'utente in un giudizio numerico.
    """
    # 1. Applica la stessa pulizia usata durante l'addestramento
    cleaned = clean_text(user_input)
    
    # 2. Vettorizzazione Manuale:
    # Trasforma i token in indici numerici basandosi sul dizionario mock_vocab.
    tokens = cleaned.split()[:max_len]
    seq = [tokenizer_dict.get(t, 1) for t in tokens] # 1 = Out Of Vocabulary
    
    # Padding: Aggiunge zeri alla fine se la frase è più corta di 20 parole.
    padded_seq = seq + [0] * (max_len - len(seq))
    
    # 3. Predizione: np.array([padded_seq]) aggiunge la dimensione del "Batch" richiesta dalla rete.
    pred = model.predict(np.array([padded_seq]), verbose=0)[0][0]
    
    # Decisione binaria basata sulla soglia standard 0.5
    sentiment = "POSITIVO" if pred > 0.5 else "NEGATIVO"
    return sentiment, pred

# Esempio pratico di simulazione
mock_vocab = {"excellent": 10, "movie": 5, "waste": 20, "time": 30} 
input_utente = "The movie was excellent and not a waste of time!"
label, score = predict_sentiment(input_utente, model, mock_vocab)

print(f"\nInput: {input_utente} \nSentiment: {label} (Confidenza: {score:.2f})")

[INFO] Modello salvato correttamente in formato .keras

Input: The movie was excellent and not a waste of time! 
Sentiment: POSITIVO (Confidenza: 0.50)


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\keras\src\backend\torch\rnn.py:656: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1479.)
  outputs, h_n, c_n = torch._VF.lstm(
